# SmartVision training on Kaggle

1. Attach the **smartvision-code** dataset (Add Input). The folder name under `/kaggle/input` can vary.
2. Turn **Internet** on so the requirements and Roboflow downloads can run.
3. Choose a **GPU** accelerator. Training uses `device=0`, so only one GPU is used when two are attached.
4. Add a secret named `ROBOFLOW_API_KEY` (Add-ons → Secrets). Do not paste the key into a cell.


In [ ]:
import os
import shutil
import sys
from pathlib import Path

input_root = Path("/kaggle/input")
destination = Path("/kaggle/working/smartvision")

matches = [
    path.parents[1]
    for path in input_root.rglob("prepare_dataset.py")
    if path.parent.name == "scripts" and path.name == "prepare_dataset.py"
]
if not matches:
    print(input_root)
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            print(path.relative_to(input_root))
    else:
        print("(missing)")
    raise FileNotFoundError("scripts/prepare_dataset.py was not found under /kaggle/input")

project = matches[0]
print("found", project)
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree(project, destination)
os.chdir(destination)
if str(destination) not in sys.path:
    sys.path.insert(0, str(destination))
print("project", destination)

!pip install -q -r /kaggle/working/smartvision/requirements.txt


In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")


In [ ]:
import os
import sys
from pathlib import Path

root = Path("/kaggle/working/smartvision")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.prepare_dataset import main

main()


In [ ]:
!free -h
!nvidia-smi


In [ ]:
import os
import sys
from pathlib import Path

from ultralytics import YOLO

root = Path("/kaggle/working/smartvision")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

if not getattr(YOLO.train, "_smartvision_resume", False):
    _train = YOLO.train

    def train(self, *args, **kwargs):
        kwargs["cache"] = False
        project = Path(kwargs["project"])
        name = str(kwargs.get("name", "train"))
        last = project / name / "weights" / "last.pt"
        if last.is_file() and not kwargs.get("resume"):
            print(f"resuming from {last}")
            kwargs["resume"] = str(last)
        return _train(self, *args, **kwargs)

    train._smartvision_resume = True
    YOLO.train = train

from scripts.train import main

main([
    "--device", "0",
    "--workers", "2",
    "--batch", "16",
    "--epochs", "50",
    "--project", "/kaggle/working/runs",
    "--name", "train",
    "--weights-out", "/kaggle/working/best.pt",
])


In [ ]:
import os
import sys
from pathlib import Path

root = Path("/kaggle/working/smartvision")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.evaluate import main

main([
    "--weights", "/kaggle/working/best.pt",
    "--device", "0",
    "--split", "test",
    "--batch", "16",
    "--csv", "/kaggle/working/eval_results.csv",
    "--markdown", "/kaggle/working/eval_results.md",
])
